Lets Decode AI logo Side name cropped (1).jpg

# 🧠 Blueprint: How We Will Build Our Own LLM

This full course will follow the real pretraining pipeline used in modern models
like Gemma, LLaMA, and GPT-style architectures.

---

## **1. Data Preparation**
- Download TinyStories
- Clean text
- Train/validation split
- Save clean text

---

## **2. Tokenization**
- Use GPT2 BPE tokenizer
- Convert text → token IDs
- Save as memory-efficient `.bin` files

---

## **3. Creating Input–Output Pairs**
- Define block size (context length)
- Create next-token prediction labels

---

## **4. Model Architecture (Gemma-Style)**
- Token embeddings (dim=640)
- 18 Transformer blocks
- RMSNorm everywhere
- Multi-Query Attention (MQA)
- Sliding Window Attention (512 window)
- RoPE
- GLU-based FFN

---

## **5. Training the Model**
- AdamW
- Warmup + cosine decay learning rate
- Mixed precision (FP16)
- Gradient accumulation
- Checkpointing

---

## **6. Inference & Deployment**
- Autoregressive text generation
- Temperature, top-k, top-p sampling
- Save/load weights
- Upload to Hugging Face Hub

---

## 🎯 Final Goal
**A fully functional 270M parameter Small Language Model
that YOU built end-to-end.**


## 1. Check Python Environment + GPU

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — training will be extremely slow.")


PyTorch version: 2.9.0+cpu
CUDA available: False
No GPU detected — training will be extremely slow.


## 2. Install Required Dependencies

In [ ]:
!pip install datasets tiktoken numpy tqdm matplotlib einops accelerate

## 3. Create Project Folder Structure

In [ ]:
import os

base_path = "/content/drive/MyDrive/Courses/GenAI/SLM from Scratch/"

In [ ]:
folders = [
    "data",
    "data/raw",
    "data/tokenized",
    "model",
    "model/checkpoints",
    "model/logs",
    "model/config",
    "notebooks_output"
]

for f in folders:
    os.makedirs(base_path+f, exist_ok=True)

folders

['data',
 'data/raw',
 'data/tokenized',
 'model',
 'model/checkpoints',
 'model/logs',
 'model/config',
 'notebooks_output']

## 4. Load a Preview of the TinyStories Dataset

In [ ]:
from datasets import load_dataset

print("Loading TinyStories preview...")
ds_stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

# Take only 5 samples
preview = []
for i, sample in enumerate(ds_stream):
    preview.append(sample["text"])
    if i >= 4:
        break

preview


Loading TinyStories preview...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that we

# DATA PREPARATION

In [ ]:
import os
from datasets import load_dataset
from tqdm import tqdm

In [ ]:
# Defining base paths for train and val data
train_out = base_path + "data/raw/train.txt"
val_out   = base_path + "data/raw/val.txt"

## Cleaning Function

In [ ]:
def clean_text(txt: str):
    if txt is None:
        return None

    txt = txt.strip()

    # Skip empty or extremely short strings
    if len(txt) < 10:
        return None

    return txt

## Load TinyStories in Streaming Mode

In [ ]:
train_stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
val_stream   = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)

## Save Cleaned Training Text (Streaming)

In [ ]:
if os.path.exists(train_out):
    print(f"Found existing {train_out}. Skipping download & clean.")
else:
    print(f"Cleaning Training Data...")
    with open(train_out, "w") as f:
        for sample in tqdm(train_stream, desc="Processing Train"):
            txt = clean_text(sample["text"])
            if txt: f.write(txt + "\n")

Found existing /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/raw/train.txt. Skipping download & clean.


## Save Cleaned Validation Text (Streaming)

In [ ]:
if os.path.exists(val_out):
    print(f"Found existing {val_out}. Skipping download & clean.")
else:
    print(f"Cleaning Validation Data...")
    with open(val_out, "w") as f:
        for sample in tqdm(val_stream, desc="Processing Val"):
            txt = clean_text(sample["text"])
            if txt: f.write(txt + "\n")

Found existing /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/raw/val.txt. Skipping download & clean.


# TOKENIZATION + INPUT/OUTPUT BATCHES

In [ ]:
import os
import numpy as np
from tqdm import tqdm
import tiktoken

In [ ]:
# Base train and val paths to store tokenized data
train_bin_path = base_path + "data/tokenized/train.bin"
val_bin_path   = base_path + "data/tokenized/val.bin"

## Load Cleaned Text File Paths

In [ ]:
train_path = base_path+"data/raw/train.txt"
val_path   = base_path+"data/raw/val.txt"

assert os.path.exists(train_path), "train.txt missing"
assert os.path.exists(val_path),   "val.txt missing"

## Initialize GPT-2 Tokenizer

In [ ]:
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab
vocab_size

50257

## Count Total Tokens (Streaming)

We perform a first pass through the file to count how many total tokens we need to allocate

In [ ]:
def count_tokens_in_file(path):
    total = 0
    with open(path, "r") as f:
        for line in tqdm(f, desc=f"Counting tokens in {path}"):
            total += len(enc.encode(line.strip()))
    return total

## Create Memory-Mapped Files (.bin)

These .bin files store all token IDs, RAM-free.

Second Pass: Tokenize & Write Directly to Disk

In [ ]:
def prepare_bin_file(txt_path, bin_path, desc):
    """
    Checks if bin file exists. If yes, calculates shape from file size.
    If no, counts tokens, creates file, and fills it.
    Returns: The memory-mapped array.
    """
    if os.path.exists(bin_path):
        # Calculate tokens from file size (uint16 = 2 bytes)
        file_size_bytes = os.path.getsize(bin_path)
        total_tokens = file_size_bytes // 2
        print(f"Found {bin_path}. Loading {total_tokens} tokens directly...")

        # Load in Read-Only mode
        data = np.memmap(bin_path, dtype=np.uint16, mode='r', shape=(total_tokens,))
        return data

    print(f"Tokenizing {desc} (First Run)...")

    # 1. Count Tokens
    total_tokens = 0
    with open(txt_path, "r") as f:
        for line in tqdm(f, desc=f"Counting {desc}"):
            total_tokens += len(enc.encode(line.strip()))

    print(f"Allocating {total_tokens} tokens...")

    # 2. Create Memmap
    mmap = np.memmap(bin_path, dtype=np.uint16, mode='w+', shape=(total_tokens,))

    # 3. Write Tokens
    idx = 0
    with open(txt_path, "r") as f:
        for line in tqdm(f, desc=f"Writing {desc}"):
            ids = enc.encode(line.strip())
            mmap[idx : idx + len(ids)] = ids
            idx += len(ids)

    mmap.flush()

    # Return as Read-Only for safety
    return np.memmap(bin_path, dtype=np.uint16, mode='r', shape=(total_tokens,))

# --- Execute Pipeline ---
train_data = prepare_bin_file(train_out, train_bin_path, "Train")
val_data   = prepare_bin_file(val_out, val_bin_path, "Val")

print(f"\nFinal Training Tokens: {len(train_data)}")
print(f"Final Validation Tokens: {len(val_data)}")

Found /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/tokenized/train.bin. Loading 451644797 tokens directly...
Found /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/data/tokenized/val.bin. Loading 4545541 tokens directly...

Final Training Tokens: 451644797
Final Validation Tokens: 4545541


## Input–Output Batch Generator

In [ ]:
# --- Hyperparameters for Data ---
block_size = 512      # Context length (how far back the model looks)
batch_size = 8       # How many sequences to process in parallel
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(split):
    # Select the correct dataset
    data = train_data if split == 'train' else val_data

    # Randomly select starting indices for the batch
    # We subtract block_size to avoid going out of bounds
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Stack sequences to form inputs (x) and targets (y)
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    # Move to GPU
    if device == 'cuda':
        x = x.pin_memory().to(device, non_blocking=True) # pin_memory locks the memory of the tensor in RAM. This allows faster transfer of tensor to the GPU
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y

## Test the Batch Loader

In [ ]:
xb, yb = get_batch('train')
print(f"Device: {device}")
print(f"Input shape: {xb.shape}")   # Should be [8, 512]
print(f"Target shape: {yb.shape}")  # Should be [8, 512]

Device: cpu
Input shape: torch.Size([8, 512])
Target shape: torch.Size([8, 512])


# MODEL ARCHITECTURE (GEMMA-STYLE SLM)

This is the core. We will build a Gemma-style Transformer:

- RoPE (Rotary Positional Embeddings): Encodes position mathematically.

- RMSNorm: Faster, simpler normalization.

- MQA (Multi-Query Attention): Uses 1 Key/Value head for all Query heads (faster inference).

- Sliding Window Attention: Saves memory by attending only to recent tokens.

- GeGLU FeedForward: A higher-quality activation function.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

## RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        var = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(var + self.eps)
        return self.weight * x_norm

## Rotary Positional Embeddings (RoPE)
Generate sin/cos matrices for RoPE:

In [ ]:
def compute_rope_params(head_dim, theta_base=10000, context_length=4096, device='cpu'):
    # Compute inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length, device=device)

    # Compute angles: (context_length, head_dim/2)
    angles = positions[:, None] * inv_freq[None, :]

    # Expand to (context_length, head_dim) by repeating
    angles = torch.cat([angles, angles], dim=1)

    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin

## Apply RoPE

In [ ]:
def apply_rope(x, cos, sin):
    # x: (batch, seq_len, num_heads, head_dim)
    # We apply RoPE on the last dimension
    head_dim = x.shape[-1]
    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]

    # Adjust cos/sin shapes for broadcasting: (1, seq_len, 1, head_dim)
    cos = cos[:x.shape[1], :].unsqueeze(0).unsqueeze(2)
    sin = sin[:x.shape[1], :].unsqueeze(0).unsqueeze(2)

    rotated = torch.cat((-x2, x1), dim=-1)
    return (x * cos) + (rotated * sin)

## Grouped Query Attention (Supports MQA & Sliding Window)

Efficient attention for SLMs.

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_heads = cfg["n_heads"]
        self.num_kv_groups = cfg["n_kv_groups"] # If 1, it's MQA
        self.head_dim = cfg["head_dim"]
        self.emb_dim = cfg["emb_dim"]

        # Dimensions
        self.q_dim = self.num_heads * self.head_dim
        self.kv_dim = self.num_kv_groups * self.head_dim

        # Projections
        self.W_q = nn.Linear(self.emb_dim, self.q_dim, bias=False)
        self.W_k = nn.Linear(self.emb_dim, self.kv_dim, bias=False)
        self.W_v = nn.Linear(self.emb_dim, self.kv_dim, bias=False)
        self.out_proj = nn.Linear(self.q_dim, self.emb_dim, bias=False)

    def forward(self, x, cos, sin, mask=None):
        batch, seq_len, _ = x.shape

        # 1. Project Q, K, V
        q = self.W_q(x).view(batch, seq_len, self.num_heads, self.head_dim)
        k = self.W_k(x).view(batch, seq_len, self.num_kv_groups, self.head_dim)
        v = self.W_v(x).view(batch, seq_len, self.num_kv_groups, self.head_dim)

        # 2. Apply RoPE
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # 3. Repeat K/V for GQA/MQA (Broadcasting preparation)
        # If num_kv_groups < num_heads, we repeat K/V to match heads
        groups_per_head = self.num_heads // self.num_kv_groups
        k = k.repeat_interleave(groups_per_head, dim=2)
        v = v.repeat_interleave(groups_per_head, dim=2)

        # 4. Attention mechanism
        # Transpose for dot product: (batch, num_heads, seq_len, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scores: (batch, heads, seq_len, seq_len)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        probs = F.softmax(scores, dim=-1)
        output = probs @ v

        # 5. Output projection
        output = output.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.out_proj(output)

## Feed-Forward Network with GLU

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False) # Gate
        self.w2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False) # Value
        self.w3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], bias=False) # Output

    def forward(self, x):
        return self.w3(F.gelu(self.w1(x)) * self.w2(x))

## Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg["emb_dim"])
        self.attn = GroupedQueryAttention(cfg)
        self.norm2 = RMSNorm(cfg["emb_dim"])
        self.ffn = FeedForward(cfg)

    def forward(self, x, cos, sin, mask):
        # Pre-norm architecture
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        x = x + self.ffn(self.norm2(x))
        return x

## Full Gemma-Style SLM Model

In [ ]:
class Gemma3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # Embeddings
        self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        # Blocks
        self.layers = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # Final Norm
        self.final_norm = RMSNorm(cfg["emb_dim"])

        # Output Head (Weight Tying is common, but we'll use a separate layer for simplicity)
        self.lm_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

        # Precompute RoPE Cache
        cos, sin = compute_rope_params(cfg["head_dim"], context_length=cfg["context_length"])
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def create_mask(self, seq_len, sliding_window=None):
        # Create standard causal mask (lower triangular)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=self.cos.device))

        # Apply sliding window if requested (zero out tokens too far in the past)
        if sliding_window is not None:
            # Create a band mask
            band_mask = torch.triu(torch.ones(seq_len, seq_len, device=self.cos.device), diagonal=-sliding_window)
            mask = mask * band_mask

        return mask.unsqueeze(0).unsqueeze(0) # (1, 1, seq_len, seq_len)

    def forward(self, idx, targets=None):
        batch, seq_len = idx.shape
        x = self.token_emb(idx)

        # RoPE cache slicing
        cos = self.cos[:seq_len, :]
        sin = self.sin[:seq_len, :]

        # Process Layers
        for i, layer in enumerate(self.layers):
            # Check layer type for masking (Sliding vs Full)
            layer_type = self.cfg["layer_types"][i]
            window = self.cfg["sliding_window"] if layer_type == "sliding_attention" else None
            mask = self.create_mask(seq_len, window)

            x = layer(x, cos, sin, mask)

        x = self.final_norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        else:
            # Inference optimization: only calculate logits for the last token
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

## Model Configuration

In [ ]:
GEMMA3_CONFIG_270M = {
    "vocab_size": vocab_size,
    "context_length": block_size, # Train on 512 context for speed
    "emb_dim": 896,               # Increased dim to hit ~270M params
    "n_heads": 8,
    "head_dim": 112,              # 896 / 8 = 112
    "n_layers": 18,
    "hidden_dim": 896 * 4,        # Standard expansion
    "n_kv_groups": 1,             # Multi-Query Attention (MQA)
    "sliding_window": 256,        # Local attention window
    "layer_types": [              # Alternating Local/Global
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention", # Every 4th layer global
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention",
        "full_attention",
        "sliding_attention", "full_attention"
    ]
}

## Instantiate Model

In [ ]:
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
model = Gemma3Model(GEMMA3_CONFIG_270M)
model.to(device)

# Calculate Parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model Parameters: {total_params / 1e6:.2f} Million")

Model Parameters: 296.02 Million


## TRAINING THE GEMMA-STYLE SLM

We will break this into three parts for clarity:

- Evaluation Helper: A function to check how well the model is doing without updating weights.

- Optimizer & Scheduler: The algorithms that adjust the weights.

- The Main Loop: The actual training process.

### The Evaluation Helper (estimate_loss)
Training loss fluctuates wildly because we look at small batches. To get a true picture of progress, we periodically pause training and evaluate the model on both the training and validation sets without updating gradients.

In [ ]:
@torch.no_grad() # Tell PyTorch: "Don't store gradients, we are just looking!"
def estimate_loss(model, eval_iters=100):
    out = {}
    model.eval() # Switch model to 'eval' mode (turns off Dropout, etc.)

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train() # Switch back to 'train' mode
    return out

### Optimizer & Scheduler Setup
We use AdamW, the gold standard for LLMs. We also use a Learning Rate Scheduler.

- Warmup: We start with a low learning rate and increase it linearly. This stabilizes early training.

- Cosine Decay: We slowly lower the learning rate. This helps the model settle into a good solution ("minima") at the end.

In [ ]:
import time
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

In [ ]:
# --- Training Hyperparameters ---
max_iters = 5000       # Total training steps (batches)
eval_interval = 250    # How often to check validation loss
save_interval = 500   # How often to save a checkpoint
learning_rate = 3e-4   # Peak learning rate (standard for this size)
warmup_steps = 200     # Steps to reach peak LR
min_lr = 3e-5          # Minimum LR (10% of peak)
grad_accum_steps = 16   # Simulate larger batch size (8 * 16 = 128 effective batch)
batch_size = 8          # Fits easily in T4 Memory

#### Initialize Optimizer

In [ ]:
# 1. Optimizer: AdamW
# We separate parameters that need weight decay (weights) from those that don't (biases, norms)
# This is a standard practice to improve generalization.
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

optim_groups = [
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': nodecay_params, 'weight_decay': 0.0}
]

optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95))

In [ ]:
# 2. Scheduler: Warmup -> Cosine Decay
scheduler_warmup = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=min_lr)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])

print("Optimizer and Scheduler ready!")

Optimizer and Scheduler ready!


In [ ]:
print("Compiling model for faster training on T4... (This takes a minute at start)")
model = torch.compile(model)

Compiling model for faster training on T4... (This takes a minute at start)


## TRAINING LOOP

This loop includes Mixed Precision (torch.amp). This allows the model to calculate some operations in float16 or bfloat16 (faster, less memory) while keeping weights in float32 for stability.

In [ ]:
from contextlib import nullcontext

# --- Setup for Mixed Precision ---
# bfloat16 is preferred on Ampere GPUs (A100, A10, 3090/4090)
# float16 is standard for older GPUs (T4, V100)
dtype = 'float16'
ptdtype = torch.float16

# The Scaler handles the math conversion stability for float16
scaler = torch.amp.GradScaler('cuda', enabled=True)
ctx = torch.amp.autocast(device_type='cuda', dtype=ptdtype)

print(f"Starting training with {dtype} precision...")
print(f"Effective Batch Size: {batch_size * grad_accum_steps}")

# Initialize
checkpoint_path = base_path + "model/checkpoints/best_model.pt"
start_iter = 0
t0 = time.time()
best_val_loss = float('inf')
train_losses = []
val_losses = []

if os.path.exists(checkpoint_path):
    print(f"Checkpoint found at {checkpoint_path}")
    print("Loading weights to resume training...")

    # Load Weights
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)

    # Verify performance immediately to set best_val_loss correctly
    # (This prevents overwriting a good model with a bad one in the first step)
    print("   Verifying checkpoint performance (this takes a moment)...")
    initial_losses = estimate_loss(model)
    best_val_loss = initial_losses['val']
    print(f"Resumed successfully! Previous Best Val Loss: {best_val_loss:.4f}")
else:
    print("No checkpoint found. Starting training from scratch.")

for iter in range(max_iters):

    # --- 1. Evaluation & Logging ---
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        dt = time.time() - t0
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, time {dt:.2f}s")

        # Log for plotting later
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])

        # Save best model
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), base_path + "model/checkpoints/best_model.pt")
            print(f"Saved new best model (Loss: {best_val_loss:.4f})")
        t0 = time.time() # Reset timer

    # --- 2. Forward & Backward Pass (with Gradient Accumulation) ---
    # We iterate multiple times (micro-batches) before updating weights once.
    # This allows us to train with "large batches" on small GPUs.
    for micro_step in range(grad_accum_steps):
        X, Y = get_batch('train')
        if X.shape[0] != batch_size: X, Y = X[:batch_size], Y[:batch_size]

        # Mixed Precision Forward Pass
        with ctx:
            logits, loss = model(X, Y)
            # Scale loss because we are summing gradients over multiple steps
            loss = loss / grad_accum_steps

        # Mixed Precision Backward Pass
        # Scaler ensures small gradients don't "vanish" to zero in float16
        scaler.scale(loss).backward()

    # --- 3. Optimizer Step ---
    # Unscale gradients to apply clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Prevents "exploding gradients"

    # Update weights
    scaler.step(optimizer)
    scaler.update()

    # Clear gradients for the next step
    # set_to_none=True is slightly faster than zero_grad()
    optimizer.zero_grad(set_to_none=True)

    # --- 4. Scheduler Step ---
    scheduler.step()

print("Training Complete! Best model saved.")

Starting training with float16 precision...
Effective Batch Size: 128


/usr/local/lib/python3.12/dist-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()
W0105 18:43:31.465000 11688 torch/_inductor/utils.py:1558] [0/0] Not enough SMs to use max_autotune_gemm mode


step 0: train loss 11.0016, val loss 11.0005, time 189.93s
Saved new best model (Loss: 11.0005)
step 250: train loss 2.6008, val loss 2.5862, time 2112.60s
Saved new best model (Loss: 2.5862)
step 500: train loss 2.0612, val loss 2.0502, time 2033.90s
Saved new best model (Loss: 2.0502)
step 750: train loss 1.8573, val loss 1.8738, time 2032.49s
Saved new best model (Loss: 1.8738)
step 1000: train loss 1.7339, val loss 1.7950, time 2028.69s
Saved new best model (Loss: 1.7950)
step 1250: train loss 1.6834, val loss 1.7008, time 2027.42s
Saved new best model (Loss: 1.7008)
step 1500: train loss 1.6399, val loss 1.6546, time 2004.60s
Saved new best model (Loss: 1.6546)
step 1750: train loss 1.5678, val loss 1.5961, time 2028.30s
Saved new best model (Loss: 1.5961)
step 2000: train loss 1.5213, val loss 1.5676, time 2028.40s
Saved new best model (Loss: 1.5676)


In [ ]:
import matplotlib.pyplot as plt
# Plot the training curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Steps")
plt.ylabel("Loss")
plt.title("SLM Training Progress")
plt.legend()
plt.grid(True)
plt.show()

# INFERENCE + SAMPLING + DEPLOYMENT

In this phase, we switch the model from "Learning Mode" to "Generation Mode". We define a function that takes a text prompt, feeds it to the model, and asks it to predict one token at a time until it completes a story.

In [ ]:
import torch
import torch.nn.functional as F
import tiktoken
from tqdm import tqdm

## Load the Trained Model
We need to instantiate a fresh model architecture and then load the specific weights we saved during training.



In [ ]:
# --- 1. Load Model Architecture ---
# We must use the EXACT same config as training
print("Initializing model structure...")
model = Gemma3Model(GEMMA3_CONFIG_270M)
model.to(device)

# --- 2. Load Trained Weights ---
# We load the weights from the 'best_model.pt' file we saved
checkpoint_path = base_path + "model/checkpoints/best_model.pt"

if os.path.exists(checkpoint_path):
    print(f"Loading weights from {checkpoint_path}...")
    state_dict = torch.load(checkpoint_path, map_location=device)
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    print("Model loaded successfully!")
else:
    print("No checkpoint found! Using random weights (Output will be gibberish).")

# --- 3. Switch to Eval Mode ---
# This disables dropout and other training-specific behaviors
model.eval()


Initializing model structure...
Loading weights from /content/drive/MyDrive/Courses/GenAI/SLM from Scratch/model/checkpoints/best_model.pt...
Model loaded successfully!


Gemma3Model(
  (token_emb): Embedding(50257, 896)
  (layers): ModuleList(
    (0-17): 18 x TransformerBlock(
      (norm1): RMSNorm()
      (attn): GroupedQueryAttention(
        (W_q): Linear(in_features=896, out_features=896, bias=False)
        (W_k): Linear(in_features=896, out_features=112, bias=False)
        (W_v): Linear(in_features=896, out_features=112, bias=False)
        (out_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (norm2): RMSNorm()
      (ffn): FeedForward(
        (w1): Linear(in_features=896, out_features=3584, bias=False)
        (w2): Linear(in_features=896, out_features=3584, bias=False)
        (w3): Linear(in_features=3584, out_features=896, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_head): Linear(in_features=896, out_features=50257, bias=False)
)

## Define the Generator Function
1. This function implements Autoregressive Generation:

2. Input: "Once upon a time"

3. Model Predicts: "there"

4. New Input: "Once upon a time there"

5. Model Predicts: "was"

6. Repeat...

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    """
    Generates text from a prompt.

    Args:
        prompt (str): The starting text.
        max_new_tokens (int): How many words to write.
        temperature (float): Creativity control.
                             Low (0.2) = Focused/Repetitive.
                             High (0.9) = Creative/Random.
        top_k (int): Quality control. Only considers the top K most likely words.
    """

    # 1. Encode the Prompt
    # Convert string "Hello" -> [15496] (Token IDs)
    input_ids = enc.encode(prompt)

    # Convert to Tensor and move to GPU
    # Shape: (1, seq_len) -> Batch size of 1
    idx = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

    # 2. Generation Loop
    for _ in range(max_new_tokens):
        # Crop context if it gets too long (we only trained on 'block_size' length)
        # If history is 600 tokens and block_size is 512, keep last 512
        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]

        # Forward Pass
        # We only care about the logits for the very last token
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] # (Batch, Vocab_Size)

        # 3. Apply Temperature
        # Higher T makes differences between probabilities smaller (more random)
        logits = logits / temperature

        # 4. Top-K Sampling
        # Keep only the top K probabilities, set the rest to -infinity
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        # 5. Sample the Next Token
        probs = F.softmax(logits, dim=-1)

        # random selection based on probabilities
        idx_next = torch.multinomial(probs, num_samples=1)

        # 6. Append to History
        idx = torch.cat((idx, idx_next), dim=1)

    # 7. Decode
    # Convert List of Ints -> String
    output = idx[0].tolist()
    return enc.decode(output)

## Test Generation

Now, let's see what your model has learned. Since we trained on TinyStories, it should be good at generating simple, coherent stories for children.

In [ ]:
# --- Test 1: Standard Story ---
print("-" * 50)
prompt1 = "Once upon a time, there was a little girl named Lily."
print(generate(prompt1, max_new_tokens=150, temperature=0.7))

# --- Test 2: High Creativity (Higher Temperature) ---
print("\n" + "-" * 50)
prompt2 = "One day, a big red dragon flew"
print(generate(prompt2, max_new_tokens=150, temperature=0.9))

# --- Test 3: Focused (Lower Temperature) ---
print("\n" + "-" * 50)
prompt3 = "The cat sat on the mat and"
print(generate(prompt3, max_new_tokens=100, temperature=0.4))

--------------------------------------------------
Once upon a time, there was a little girl named Lily. She loved to play with her toys and have fun with her friends. One day, she found a big, shiny fork in the park. She picked it up and started to play with it.As she was playing, she heard a loud noise. It was coming from the other side of the park. She looked up and saw a big truck. The truck was very big and had lots of fuel. Lily was scared at first, but then she saw that the truck was making a lot of noise.Lily decided to be brave and play with the truck. She threw the truck at the truck and it hit her. She laughed and had fun. At the end of the day, Lily knew that the truck was not scary at all. She

--------------------------------------------------
One day, a big red dragon flew into the sky. It was so big she couldn't see what happened to it. She started to feel scared and she froze.The dragon looked around and saw a kind rabbit. He asked her, "What's wrong?"The rabbit said, 

In [ ]:
# --- Test 2: High Creativity (Higher Temperature) ---
print("\n" + "-" * 50)
prompt2 = "One day, a boy was making a video"
print(generate(prompt2, max_new_tokens=150, temperature=0.9))


--------------------------------------------------
One day, a boy was making a video. When the boy cut out the pieces, he was so proud. The video was so hard to see out, so he asked his dad to help him. The dad said yes and showed him how to make the video right.But then, something bad happened. The boy made a too serious mistake and couldn't stop the film. He felt really worried, but he was also very worried about the big mess he made.When the film was over, the brother was very disappointed. He had to go home without his dad. He knew he had to be more careful with the video now and that it was important to be careful with his work. He had learned his lesson.Once upon a time there were two best friends named Jack and Jill.
